##  Presidio 

**Presidio** is an open-source framework used to **detect and protect sensitive information (PII)** in text, such as phone numbers, email addresses, names, and IDs.

It is commonly used in:
- Chatbots and AI applications
- Log sanitization
- Compliance workflows (GDPR, HIPAA)
- Pre-processing data before sending it to LLMs

###  What Presidio Does
- **Analyzes text** to find sensitive entities (PII)
- **Classifies** the type of data (e.g., PHONE_NUMBER, EMAIL)
- **Returns metadata** like position and confidence score
- Can optionally **mask or anonymize** detected data

###  Key Components
- **AnalyzerEngine** – detects sensitive data in text  
- **Recognizers** – rules (regex, deny-lists, patterns) that define *how* to detect PII  
- **Language support** – recognizers run based on the specified language (e.g., `en`)

### 🧠 Typical Flow
Text → AnalyzerEngine → PII Detection → (Optional) Anonymization


###  Why Use Presidio?
- Fast and deterministic
- Works without LLMs
- Easy to extend with custom rules
- Production-ready for security and compliance


In [3]:
!pip install presidio_analyzer presidio_anonymizer
!python -m spacy download en_core_web_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 34.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 5.8 MB/s eta 0:00:00
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl (400.7 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Analyzer with Pattern recognizer with entity parameter

In [13]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer

# Text to analyze
text_to_recognize = "His name is Mr. Jones and his phone number is 212-555-5555"

# Create analyzer engine
analyzer = AnalyzerEngine()

# Run analysis
# IMPORTANT: entities must be a list, not a single string
results = analyzer.analyze(
    text=text_to_recognize,
    entities=["PHONE_NUMBER"],
    language="en"
)

print(results)


[type: PHONE_NUMBER, start: 46, end: 58, score: 0.75]


### Analyzer with Pattern recognizer without entity parameter 

In [15]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer

# Text to analyze
text_to_recognize = "His name is Mr. Jones and his phone number is 212-555-5555"

# Create analyzer engine
analyzer = AnalyzerEngine()

# Run analysis
# IMPORTANT: entities must be a list, not a single string
results = analyzer.analyze(
    text=text_to_recognize,
    language="en"
)

print(results)


[type: PERSON, start: 16, end: 21, score: 0.85, type: PHONE_NUMBER, start: 46, end: 58, score: 0.75]


### Analyzer with Pattern recognizer using deny list


In [11]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer

# Text to analyze
text_to_recognize = "His name is Mr. Jones and his phone number is 212-555-5555"

# Create analyzer engine
analyzer = AnalyzerEngine()

# Create a recognizer that detects titles using a deny list
# supported_entity is the entity label you want in results
title_recognizer = PatternRecognizer(
    supported_entity="TITLE",          # use a consistent entity name (caps is common)
    deny_list=["Mr.", "Mrs."]          # exact strings to match
)

# Register the recognizer
analyzer.registry.add_recognizer(title_recognizer)

# Run analysis
# IMPORTANT: entities must be a list, not a single string
results = analyzer.analyze(
    text=text_to_recognize,
    entities=["TITLE"],
    language="en"
)

print(results)


[type: Title, start: 12, end: 15, score: 1.0]


### Analyzer with Pattern recognizer using deny list

In [ ]:
# Import Presidio components
# AnalyzerEngine  -> main engine that runs entity detection
# PatternRecognizer -> recognizer based on regex / deny list patterns
# Pattern -> defines a single regex pattern + confidence score
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern


# Text we want to analyze
text = "His name is Mr.Jones and his phone number is 212-555-5555"


# ---------------------------------------------------
# Create the analyzer engine
# This engine manages ALL recognizers (built-in + custom)
# ---------------------------------------------------
analyzer = AnalyzerEngine()


# ---------------------------------------------------
# Define regex patterns for detecting titles
# ---------------------------------------------------
patterns = [

    # Pattern 1:
    # Matches "Mr." or "Mrs." when treated as a word
    # Example: "Mr. Jones"
    Pattern(
        name="title",                     # Pattern name (for debugging/logs)
        regex=r"\b(Mr|Mrs)\.\b?",          # Regular expression
        score=0.8                          # Confidence score (80%)
    ),

    # Pattern 2:
    # Matches "Mr." or "Mrs." when NO SPACE exists
    # Example: "Mr.Jones"
    Pattern(
        name="title_no_space",
        regex=r"\b(Mr|Mrs)\.(?=[A-Z])",    # Lookahead for capital letter
        score=0.8
    ),
]


# ---------------------------------------------------
# Create a PatternRecognizer
# ---------------------------------------------------
title_recognizer = PatternRecognizer(
    supported_entity="Title",   # Entity label returned in results
    patterns=patterns           # Regex rules used for detection
)


# ---------------------------------------------------
# Register the recognizer with the analyzer
# ---------------------------------------------------
analyzer.registry.add_recognizer(title_recognizer)


# ---------------------------------------------------
# Run analysis
# ---------------------------------------------------
results = analyzer.analyze(
    text=text,                  # Text to scan
    entities=["Title"],          # Only look for Title entities
    language="en"                # Language context (English)
)

# Print detected entities
print(results)


[type: Title, start: 12, end: 15, score: 0.8]
